# EMERALD-JAX on Craftax — speed benchmark + 10M training

1. Verify the **real GPU throughput** (`craftax()` vs `craftax_fast()`, clean steady-state past compile).
2. Project the 10M wall-clock.
3. Launch the full **10M** run with Drive checkpointing + auto-resume.

**Before running:** Runtime → Change runtime type → **A100 GPU** (any GPU works; A100 fastest).

In [ ]:
# 1. GPU
!nvidia-smi -L

In [ ]:
# 2. Code (latest, incl. fixed-buffer imagine + craftax_fast)
%cd /content
!git clone https://github.com/JuliusSamwer/Nexus_v1.git 2>/dev/null || (cd Nexus_v1 && git pull -q)
%cd /content/Nexus_v1
!git log --oneline -1

In [ ]:
# 3. Install (craftax pins, then CUDA jax on top)
!pip install -q craftax flax optax
!pip install -q -U "jax[cuda12]"
print('installed')

In [ ]:
# 4. Verify JAX sees the GPU
import jax
print('jax', jax.__version__, '| devices:', jax.devices())
assert any('cuda' in d.platform.lower() or d.platform=='gpu' for d in jax.devices()), \
    'No GPU visible to JAX — set Runtime → A100 GPU and re-run cells 3-4.'

In [ ]:
# 5. Steady-state benchmark: warm up past the (one-time) JIT compile, THEN time.
#    This is the honest env/s — not the compile-polluted cumulative number train.run prints.
import time, jax
from emerald_jax import config, train, replay

def bench(cfg, collect_steps=64, grad_per_collect=1, warmup=2, timed=6, label=''):
    st = train.init_state(cfg, seed=0)
    rollout = train.make_rollout(st['agent'], st['env'], st['eparams'], st['ach_keys'], st['A'])
    train_step = train.make_train_step(st['agent'], st['tx'], cfg)
    n_grad = max(1, int(grad_per_collect*collect_steps))
    def collect():
        (st['obs'], st['estate'], st['rings'], st['key']), outs = rollout(
            st['params'], st['obs'], st['estate'], st['rings'], st['key'], collect_steps, True)
        img,a_int,reward,done,_ = outs
        st['buf'] = replay.add_rollout(st['buf'], img, a_int, reward, done)
        st['env_step'] += collect_steps*cfg.num_envs
    def grads():
        for _ in range(n_grad):
            st['key'], bk, tk = jax.random.split(st['key'],3)
            batch = replay.sample(st['buf'], bk, cfg.batch_size, cfg.L, st['A'])
            st['params'], st['opt_state'], st['perc'], m = train_step(
                st['params'], st['opt_state'], st['perc'], batch, tk)
            st['grad_step'] += 1
    while st['env_step'] < max(cfg.prefill, cfg.L*cfg.num_envs):
        collect()
    for _ in range(warmup):   # trigger + discard compile
        collect(); grads()
    jax.block_until_ready(st['params'])
    e0,g0 = st['env_step'], st['grad_step']; t0=time.time()
    for _ in range(timed):
        collect(); grads()
    jax.block_until_ready(st['params'])
    dt=time.time()-t0
    eps=(st['env_step']-e0)/dt; gps=(st['grad_step']-g0)/dt
    print(f'[{label:>12}] env/s {eps:8.0f}  grad/s {gps:6.1f}   (envs {cfg.num_envs}, batch {cfg.batch_size}, n_grad {n_grad})')
    return eps
print('bench() ready')

In [ ]:
# 6. A/B: baseline vs the throughput preset. (If craftax_fast OOMs on a small GPU,
#    lower num_envs/batch_size below — keep capacity*num_envs roughly constant.)
base = bench(config.craftax(),      label='craftax')        # 16 envs / batch 16 (~93 env/s on A100 before)
fast = bench(config.craftax_fast(), label='craftax_fast')   # 64 envs / batch 32
print(f'\nspeedup craftax_fast / craftax: {fast/base:.2f}x')
print('(push num_envs/batch_size higher to fill the card further; watch GPU memory)')

In [ ]:
# 7. Project the 10M wall-clock from the measured fast env/s
hrs = 10_000_000 / fast / 3600
print(f'measured craftax_fast: {fast:.0f} env/s')
print(f'-> 10M env-steps ≈ {hrs:.1f} h   (vs the 4090/torch baseline of ~3.5 days for 10M)')

## Train 10M
`config.craftax_fast()` is the **identical** EMERALD architecture/hparams as `craftax()` — only parallelism changes. Checkpoints to Drive every eval; re-run the train cell after any disconnect and it **resumes**.

In [ ]:
# 8. Drive (checkpoints survive disconnects)
from google.colab import drive
drive.mount('/content/drive')
import os
CKPT_DIR = '/content/drive/MyDrive/nexus/emerald_jax_craftax_10M'
os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoints ->', CKPT_DIR)

In [ ]:
# 9. Full 10M run. First call jit-compiles (~1-3 min), then logs env/s + eval score.
#    Re-run this cell after a disconnect to resume from the last Drive checkpoint.
from emerald_jax import config, train
st, ev = train.run(
    config.craftax_fast(),
    total_env_steps = 10_000_000,
    collect_steps   = 64,
    grad_per_collect= 1,           # == train_ratio 1.0 (faithful)
    eval_every_env  = 200_000,
    log_every_env   = 20_000,
    eval_steps      = 1_000,
    ckpt_dir        = CKPT_DIR,
    resume          = True,
)
print('final Crafter score:', ev['score'])